In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    TimeoutException,
    NoSuchElementException,
)
import pandas as pd
import time


URL = "https://www.olympics.com/en/milano-cortina-2026/medals"


# --------------------------------------------------
# Browser setup
# --------------------------------------------------
driver = webdriver.Chrome()
driver.get(URL)
driver.maximize_window()

wait = WebDriverWait(driver, 15)
wait.until(EC.presence_of_element_located((By.XPATH, '//div[@role="row"]')))


# --------------------------------------------------
# Generic helpers
# --------------------------------------------------
def get_all_rows():
    """Return all currently visible table rows."""
    return driver.find_elements(By.XPATH, '//div[@role="row"]')


def scroll_down(pixels=800, pause=0.3):
    """Scroll down once and return current page offset."""
    driver.execute_script(f"window.scrollBy(0, {pixels});")
    time.sleep(pause)
    return driver.execute_script("return window.pageYOffset;")


def scroll_to_top(pause=0.3):
    """Scroll back to top of page."""
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(pause)


def parse_main_country_row(row):
    """
    Parse one visible main country row.

    Expected structure:
    Rank, Country, Gold, Silver, Bronze, Total

    Returns:
        dict with parsed values and original row element
        or None if the row is not a valid country row
    """
    try:
        cells = row.find_elements(By.XPATH, './/div[@role="cell"]')
        values = [cell.text.strip() for cell in cells if cell.text.strip()]

        if len(values) < 6:
            return None

        rank, country, gold, silver, bronze, total = values[:6]

        return {
            "row": row,
            "Rank": rank,
            "Country": country,
            "Gold": gold,
            "Silver": silver,
            "Bronze": bronze,
            "Total": total,
        }

    except StaleElementReferenceException:
        return None


# --------------------------------------------------
# Phase 1: collect main country table
# --------------------------------------------------
def collect_country_table():
    """
    Scroll through the full medal page and collect the country-level table.
    Returns a DataFrame with one row per country.
    """
    country_data = []
    seen_countries = set()
    last_scroll = -1

    while True:
        rows = get_all_rows()

        for row in rows:
            parsed = parse_main_country_row(row)
            if parsed is None:
                continue

            country = parsed["Country"]

            if country not in seen_countries:
                country_data.append({
                    "Rank": parsed["Rank"],
                    "Country": parsed["Country"],
                    "Gold": parsed["Gold"],
                    "Silver": parsed["Silver"],
                    "Bronze": parsed["Bronze"],
                    "Total": parsed["Total"],
                })
                seen_countries.add(country)

        new_scroll = scroll_down(pixels=800, pause=0.3)

        if new_scroll == last_scroll:
            break
        last_scroll = new_scroll

    country_df = pd.DataFrame(country_data)

    if not country_df.empty:
        country_df["Rank_num"] = pd.to_numeric(country_df["Rank"], errors="coerce")
        country_df = (
            country_df.sort_values(["Rank_num", "Country"])
            .drop(columns=["Rank_num"])
            .reset_index(drop=True)
        )

    return country_df


# --------------------------------------------------
# Helper: find one specific country row in the live page
# --------------------------------------------------
def find_country_row(country_name, max_scrolls=60):
    """
    Scroll until a specific country row is found in the live page.

    Returns:
        parsed row dict including the Selenium row element
        or None if not found
    """
    last_scroll = -1

    for _ in range(max_scrolls):
        rows = get_all_rows()

        for row in rows:
            parsed = parse_main_country_row(row)
            if parsed is None:
                continue

            if parsed["Country"] == country_name:
                return parsed

        new_scroll = scroll_down(pixels=700, pause=0.3)

        if new_scroll == last_scroll:
            break
        last_scroll = new_scroll

    return None


# --------------------------------------------------
# Helper: expand one country row
# --------------------------------------------------
def expand_country_row(row):
    """
    Expand a country row and return:
    - expanded details row element
    - expand button element
    """
    try:
        expand_btn = row.find_element(By.XPATH, './/button[@type="button"]')
    except NoSuchElementException:
        raise Exception("Expand button not found for this row.")

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", expand_btn)
    time.sleep(0.2)

    details_id = expand_btn.get_attribute("aria-controls")
    expanded = expand_btn.get_attribute("aria-expanded")

    if expanded == "false":
        driver.execute_script("arguments[0].click();", expand_btn)
        time.sleep(0.3)

    details_row = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, details_id))
    )

    return details_row, expand_btn


# --------------------------------------------------
# Helper: collapse one country row
# --------------------------------------------------
def collapse_country_row(expand_btn):
    """Collapse the expanded country row if it is open."""
    try:
        expanded = expand_btn.get_attribute("aria-expanded")
        if expanded == "true":
            driver.execute_script("arguments[0].click();", expand_btn)
            time.sleep(0.2)
    except Exception:
        pass


# --------------------------------------------------
# Helper: parse expanded details text
# --------------------------------------------------
def parse_details_text(details_text):
    """
    Parse expanded details text in this pattern:

    Alpine Skiing
    2
    1
    1
    4
    Bobsleigh
    1
    0
    2
    3

    Returns:
        list of dicts, one per sport
    """
    lines = [line.strip() for line in details_text.splitlines() if line.strip()]

    sports = []
    i = 0

    while i + 4 < len(lines):
        sport = lines[i]
        gold = lines[i + 1]
        silver = lines[i + 2]
        bronze = lines[i + 3]
        total = lines[i + 4]

        if gold.isdigit() and silver.isdigit() and bronze.isdigit() and total.isdigit():
            sports.append({
                "Sport": sport,
                "Sport_Gold": int(gold),
                "Sport_Silver": int(silver),
                "Sport_Bronze": int(bronze),
                "Sport_Total": int(total),
            })
            i += 5
        else:
            i += 1

    return sports


# --------------------------------------------------
# Phase 2: collect sport-level medal table
# --------------------------------------------------
def collect_sport_table(country_df):
    """
    For each country, find the live row, expand it,
    extract sport-level medal details, and build a DataFrame.
    """
    all_sports = []
    scroll_to_top(pause=0.3)

    for _, crow in country_df.iterrows():
        country_name = crow["Country"]
        print(f"Processing details for: {country_name}")

        country_info = find_country_row(country_name)

        if country_info is None:
            print(f"Could not find row for {country_name}")
            continue

        try:
            details_row, expand_btn = expand_country_row(country_info["row"])
            parsed_sports = parse_details_text(details_row.text)

            if not parsed_sports:
                print(f"No sport rows extracted for {country_name}")

            for item in parsed_sports:
                all_sports.append({
                    "Rank": crow["Rank"],
                    "Country": country_name,
                    **item,
                })

            collapse_country_row(expand_btn)

        except TimeoutException:
            print(f"Timeout while extracting {country_name}")
        except Exception as e:
            print(f"Expand/read failed for {country_name}: {e}")

    sport_df = pd.DataFrame(all_sports)

    if not sport_df.empty:
        numeric_cols = ["Sport_Gold", "Sport_Silver", "Sport_Bronze", "Sport_Total"]
        sport_df[numeric_cols] = sport_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

    return sport_df


# --------------------------------------------------
# Run everything
# --------------------------------------------------
try:
    country_df = collect_country_table()
    print(f"Collected {len(country_df)} countries")
    print(country_df.head())

    country_df.to_csv("countries.csv", index=False, encoding="utf-8-sig")

    sport_df = collect_sport_table(country_df)
    print(sport_df.head(20))

    sport_df.to_csv("country_sports.csv", index=False, encoding="utf-8-sig")

    print("Done.")
    print("Saved files:")
    print("- countries.csv")
    print("- country_sports.csv")

finally:
    driver.quit()